In [1]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PRETRAIN_DATA = os.path.join(REPO_ROOT, "data", "3.pretraining", "1.dap")

import os
import torch

def load_model_weights(model_dir):
    """지정된 디렉토리에서 모델 가중치를 로드합니다 (.bin 또는 .safetensors)."""
    bin_path = os.path.join(model_dir, "pytorch_model.bin")
    # safetensors 라이브러리가 필요한 경우를 대비한 경로
    safe_path = os.path.join(model_dir, "model.safetensors")
    
    if os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")
    elif os.path.exists(safe_path):
        from safetensors.torch import load_file
        return load_file(safe_path)
    else:
        print(f"가중치 파일을 찾을 수 없습니다: {model_dir}")
        return None

def inspect_layernorm_weights(state_dict, model_name):
    """LayerNorm 계층의 키 이름과 첫 번째 임베딩 LayerNorm의 텐서 평균/분산을 출력합니다."""
    print(f"\n[{model_name} 검사 결과]")
    
    if state_dict is None:
        return
        
    # LayerNorm과 관련된 키만 추출
    ln_keys = [k for k in state_dict.keys() if "LayerNorm" in k]
    
    if not ln_keys:
        print("LayerNorm 파라미터를 찾을 수 없습니다.")
        return
        
    # 파라미터 명명 규칙 확인 (gamma/beta vs weight/bias)
    sample_key = ln_keys[0]
    if "gamma" in sample_key or "beta" in sample_key:
        print("-> 파라미터 명명 규칙: 구버전 (gamma / beta)")
        target_weight_key = 'bert.embeddings.LayerNorm.gamma'
        target_bias_key = 'bert.embeddings.LayerNorm.beta'
    else:
        print("-> 파라미터 명명 규칙: 신버전 (weight / bias)")
        target_weight_key = 'bert.embeddings.LayerNorm.weight'
        target_bias_key = 'bert.embeddings.LayerNorm.bias'
        
    # 특정 LayerNorm 텐서의 실제 수치 확인
    if target_weight_key in state_dict and target_bias_key in state_dict:
        weight_tensor = state_dict[target_weight_key]
        bias_tensor = state_dict[target_bias_key]
        
        weight_mean = weight_tensor.mean().item()
        bias_mean = bias_tensor.mean().item()
        
        print(f"-> {target_weight_key} 평균값: {weight_mean:.6f}")
        print(f"-> {target_bias_key} 평균값: {bias_mean:.6f}")
        
        # 1.0 / 0.0과 정확히 일치하는지 확인 (초기화 여부 검증)
        if round(weight_mean, 5) == 1.00000 and round(bias_mean, 5) == 0.00000:
            print("=> 경고: 텐서 값이 기본 초기화 상태(1.0 / 0.0)입니다. 가중치가 오염(초기화)되었습니다.")
        else:
            print("=> 정상: 텐서 값이 기본값이 아니며, 정상적으로 학습된 가중치입니다.")
    else:
        print(f"=> 키를 찾을 수 없습니다: {target_weight_key}")

# 경로 지정 (실제 존재하는 체크포인트 번호로 맞추어 실행)
base_model_dir = "./models/checkpoint-225000"
mid_checkpoint_dir = os.path.join(PRETRAIN_DATA, "checkpoint-156600")
final_model_dir = os.path.join(PRETRAIN_DATA, "models")

# 1. Base 모델 로드 및 검사
base_dict = load_model_weights(base_model_dir)
inspect_layernorm_weights(base_dict, "1. 원본 Base 모델 (checkpoint-225000)")

# 2. 중간 체크포인트 로드 및 검사
mid_dict = load_model_weights(mid_checkpoint_dir)
inspect_layernorm_weights(mid_dict, "2. 중간 저장 체크포인트 (checkpoint-156600)")

# 3. 최종 저장 모델 로드 및 검사
final_dict = load_model_weights(final_model_dir)
inspect_layernorm_weights(final_dict, "3. 최종 Best 저장 모델 (outputs/models)")


[1. 원본 Base 모델 (checkpoint-225000) 검사 결과]
-> 파라미터 명명 규칙: 신버전 (weight / bias)
-> bert.embeddings.LayerNorm.weight 평균값: 1.001517
-> bert.embeddings.LayerNorm.bias 평균값: 0.000022
=> 정상: 텐서 값이 기본값이 아니며, 정상적으로 학습된 가중치입니다.

[2. 중간 저장 체크포인트 (checkpoint-156600) 검사 결과]
-> 파라미터 명명 규칙: 구버전 (gamma / beta)
-> bert.embeddings.LayerNorm.gamma 평균값: 0.979717
-> bert.embeddings.LayerNorm.beta 평균값: -0.000124
=> 정상: 텐서 값이 기본값이 아니며, 정상적으로 학습된 가중치입니다.

[3. 최종 Best 저장 모델 (outputs/models) 검사 결과]
-> 파라미터 명명 규칙: 구버전 (gamma / beta)
-> bert.embeddings.LayerNorm.gamma 평균값: 0.979717
-> bert.embeddings.LayerNorm.beta 평균값: -0.000124
=> 정상: 텐서 값이 기본값이 아니며, 정상적으로 학습된 가중치입니다.


In [2]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PRETRAIN_DATA = os.path.join(REPO_ROOT, "data", "3.pretraining", "1.dap")

import os
import torch

def get_model_keys(model_dir):
    """지정된 디렉토리의 모델 가중치 파일에서 키 목록을 추출합니다."""
    bin_path = os.path.join(model_dir, "pytorch_model.bin")
    safe_path = os.path.join(model_dir, "model.safetensors")
    
    if os.path.exists(safe_path):
        from safetensors.torch import load_file
        return set(load_file(safe_path).keys())
    elif os.path.exists(bin_path):
        return set(torch.load(bin_path, map_location="cpu").keys())
    else:
        print(f"경고: {model_dir}에서 가중치 파일을 찾을 수 없습니다.")
        return set()

# 비교할 모델 경로 설정
base_model_dir = "./models/checkpoint-225000"
ckpt_model_dir = os.path.join(PRETRAIN_DATA, "checkpoint-156600")

print("1. 모델 키 추출 중...")
base_keys = get_model_keys(base_model_dir)
ckpt_keys = get_model_keys(ckpt_model_dir)

if not base_keys or not ckpt_keys:
    print("키 추출에 실패하여 검사를 중단합니다.")
else:
    print(f"-> Base 모델(checkpoint-225000) 총 파라미터 수: {len(base_keys)}개")
    print(f"-> 저장된 체크포인트(checkpoint-156600) 총 파라미터 수: {len(ckpt_keys)}개\n")

    # 2. 집합 연산을 통한 차이 분석
    only_in_base = base_keys - ckpt_keys
    only_in_ckpt = ckpt_keys - base_keys
    common_keys = base_keys & ckpt_keys

    print("==================================================")
    print(f"[Base 모델에만 존재하고 체크포인트에는 없는 키]: {len(only_in_base)}개")
    for k in sorted(only_in_base):
        print(f"  - {k}")

    print("\n==================================================")
    print(f"[체크포인트에만 존재하고 Base 모델에는 없는 키]: {len(only_in_ckpt)}개")
    for k in sorted(only_in_ckpt):
        print(f"  - {k}")

    print("\n==================================================")
    print(f"두 모델에 공통으로 존재하는 일치하는 키: {len(common_keys)}개")

1. 모델 키 추출 중...
-> Base 모델(checkpoint-225000) 총 파라미터 수: 207개
-> 저장된 체크포인트(checkpoint-156600) 총 파라미터 수: 208개

[Base 모델에만 존재하고 체크포인트에는 없는 키]: 52개
  - bert.embeddings.LayerNorm.bias
  - bert.embeddings.LayerNorm.weight
  - bert.encoder.layer.0.attention.output.LayerNorm.bias
  - bert.encoder.layer.0.attention.output.LayerNorm.weight
  - bert.encoder.layer.0.output.LayerNorm.bias
  - bert.encoder.layer.0.output.LayerNorm.weight
  - bert.encoder.layer.1.attention.output.LayerNorm.bias
  - bert.encoder.layer.1.attention.output.LayerNorm.weight
  - bert.encoder.layer.1.output.LayerNorm.bias
  - bert.encoder.layer.1.output.LayerNorm.weight
  - bert.encoder.layer.10.attention.output.LayerNorm.bias
  - bert.encoder.layer.10.attention.output.LayerNorm.weight
  - bert.encoder.layer.10.output.LayerNorm.bias
  - bert.encoder.layer.10.output.LayerNorm.weight
  - bert.encoder.layer.11.attention.output.LayerNorm.bias
  - bert.encoder.layer.11.attention.output.LayerNorm.weight
  - bert.encoder.layer.11.